# 365 Probabilidades - Dia #100
## Qual a probabilidade de você só descobrir quem é depois de escolher?

**Tipo:** Comportamental  
**Data de publicação:** 2026-09-21  
**Ferramenta:** Python  
**Decisão analisada:** Escolher um caminho antes de saber o que ele vai fazer com você  
**Hashtag:** #365Probabilidades #Dia100

---

### 📖 A História

Aos 17 anos, pede-se a alguém uma das decisões mais longas da vida. Um curso, uma profissão, às vezes uma cidade.

E pede-se isso antes de a pessoa ter assistido a uma única aula do que está escolhendo.

A pergunta que todo mundo faz depois é se a escolha foi certa. A que quase ninguém faz é outra: dá para saber quem você é antes de escolher? Ou é a escolha que, aos poucos, vai fazendo você?

---

### 📚 O Conceito: a escolha como aprendizado, e duas contas de Poisson

Stinebrickner & Stinebrickner tratam o curso em que alguém se forma como **o resultado de um processo de aprendizado**: a pessoa descobre o curso, e o próprio desempenho nele, cursando. Na entrada, ela ainda não tem a informação de que precisaria para escolher sabendo.

Do lado da permanência, o modelo do dia usa a família da Poisson. Siméon-Denis Poisson publicou em 1837 a distribuição que **conta quantos eventos acontecem em um intervalo**, quando eles chegam de forma independente e a uma taxa constante. Ela ficou famosa em 1898, quando Ladislaus von Bortkiewicz a aplicou aos soldados prussianos mortos por coice de cavalo (122 mortes em 200 observações, média de 0,61 por ano), e desse caso vem o apelido de "lei dos pequenos números".

Mas raridade **não** é condição de uso: a Poisson vale para dados de contagem em geral, e λ pode ser 0,61 ou 600. As condições que importam são a taxa constante e a independência, e é delas que saem as duas assinaturas da família: média igual à variância, e as esperas (exponencial até o primeiro evento, Gama até o k-ésimo). Quando a variância passa da média, geralmente por eventos agrupados ou taxa heterogênea, o caminho é outro, como a binomial negativa.

A família tem três membros, e os três aparecem aqui:

- **A exponencial** descreve a saída do curso, se o risco de desistir for constante no tempo. Ela não tem memória: ter aguentado três anos não muda o risco do quarto.
- **A Poisson** conta quantas etapas do curso alguém vence por ano.
- **A Gama** descreve o tempo até vencer a k-ésima etapa, porque formar não é um evento, é uma soma de etapas.

Os dois relógios correm juntos. Como nenhum deles tem memória, cada etapa é uma corrida nova contra a desistência, vencida com probabilidade λ/(λ+μ). Formar exige vencer k corridas seguidas:

$$P(\text{formar antes de sair}) = \left(\frac{\lambda}{\lambda+\mu}\right)^{k}$$

O ponto do dia é que k e λ **não são chutados**: eles são estimados pelos próprios dados de fluxo, junto com as taxas de saída.

---

### 🧮 O Modelo

1. **O que se sabe na entrada:** crenças registradas antes da primeira aula, comparadas com o destino real (Stinebrickner & Stinebrickner).
2. **A evasão comparada:** risco anual de sair em Estatística contra o grupo exatamente comparável do INEP (público, presencial, bacharelado, mesmas coortes).
3. **As oito corridas:** ajuste por **máxima verossimilhança** de um modelo de riscos competitivos (Gama para as etapas, exponencial por trechos para a saída) às contagens de formados e desistentes ano a ano. Quatro parâmetros livres: k, λ, μ₁ e μ. O ajuste é confrontado com o observado e com a estimativa independente das taxas de saída.
4. **As checagens**, definidas antes de rodar e relatadas por inteiro, inclusive a que rejeita e a que fica no limiar: aderência global, reamostragem por curso (a incerteza que importa, já que os cursos diferem muito), previsão fora da amostra, sensibilidade ao corte do acompanhamento e teste de ritmo variável das etapas. A escolha da forma da distribuição de formatura também foi testada contra Weibull, lognormal e Gama deslocada.

**Fontes:**
- Stinebrickner, R. & Stinebrickner, T. R. (2014). A Major in Science? Initial Beliefs and Final Outcomes for College Major and Dropout. *Review of Economic Studies*, 81(1), 426-472. Números conferidos na versão NBER Working Paper 19165 (2013). Berea Panel Study, turmas de 2000 e 2001, **N = 655**.
- INEP. Indicadores de Trajetória da Educação Superior, coortes de **2011, 2013 e 2015**. Cursos de Estatística (CINE 0542E01), presenciais: 33 cursos por coorte, **5.310 ingressantes**.
- INEP, mesmas coortes (2011 e 2013): todos os cursos públicos, presenciais e de bacharelado, **560.758 ingressantes**, como grupo de comparação.
- Bortkiewicz, L. von (1898). *Das Gesetz der kleinen Zahlen*. Contexto histórico da Poisson.

**Fator ×0,80:** não aplicado. O dado de Stinebrickner é a crença individual comparada ao registro administrativo da faculdade (a distância entre as duas é o próprio achado), e o INEP é registro administrativo, não autorrelato.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats, optimize

SEED = 42
rng = np.random.default_rng(SEED)

plt.rcParams['font.family'] = 'serif'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

DOURADO, VERMELHO, VERDE, CINZA = '#c9a14a', '#c0392b', '#2a8a82', '#6b6a64'

print("✅ Bibliotecas carregadas · semente", SEED)


In [ ]:
# --- DADOS DA LITERATURA ---

# =========================================================================
# 1) Stinebrickner & Stinebrickner (2014), Review of Economic Studies 81(1):426-472
#    Números conferidos na versão NBER Working Paper 19165 (2013)
#    Berea Panel Study: turmas de 2000 e 2001, N = 655 (~85% dos ingressantes)
# =========================================================================
N_berea = 655
n_destinos = 8                      # 7 grupos de curso + desistência
p_destino_real = 0.310              # prob. média dada, na entrada, ao destino que a pessoa viveu
p_esperava_acertar = 0.433          # quanto, em média, esperavam acertar
p_acaso = 1 / n_destinos            # referência de acaso: 12,5%
ciencias_mais_provavel = 0.198      # achavam ciências (inclui matemática) o destino mais provável
ciencias_entrada = 0.156            # chance média que davam a se formar em ciências
ciencias_saida = 0.074              # se formaram em ciências
desist_esperada = 0.134             # chance média que davam à própria desistência
desist_real = 0.375                 # desistiram de fato

# =========================================================================
# 2) INEP, Indicadores de Trajetória: Estatística (CINE 0542E01), presencial
#    Coortes 2011, 2013 e 2015, 33 cursos cada, contagens somadas por ano desde o ingresso.
#    risco = quantos estavam no curso no início do ano
# =========================================================================
EST = {
    2011: dict(ing=1730, risco=[1730, 1510, 1211, 987, 728, 503, 300, 160],
               des=[210, 296, 215, 202, 104, 109, 42, 45],
               con=[10, 3, 9, 57, 121, 94, 98, 48]),
    2013: dict(ing=1762, risco=[1762, 1495, 1227, 973, 714, 452, 257, 142],
               des=[267, 261, 240, 200, 146, 83, 52, 13],
               con=[0, 7, 14, 59, 116, 112, 63, 48]),
    2015: dict(ing=1818, risco=[1818, 1550, 1216, 967, 678, 450, 305, 215],
               des=[268, 333, 242, 205, 110, 43, 32, 34],
               con=[0, 1, 7, 83, 118, 102, 58, 31]),
}
COORTES = list(EST)
T = 8                                # anos de acompanhamento usados

# =========================================================================
# 3) INEP, mesmas coortes: GRUPO COMPARÁVEL
#    todos os cursos públicos, presenciais, de bacharelado do país
#    (o número nacional de 60% de desistência mistura privada, EAD, licenciatura
#     e tecnológico, então não serve de régua para um bacharelado público)
#    risco = (desistências no ano, alunos no início do ano), anos 1 a 4
# =========================================================================
COMP = {
    2011: dict(ing=273986, des8=124166, con8=132617,
               risco=[(17333, 273986), (29112, 255583), (23516, 225059), (20260, 199071)]),
    2013: dict(ing=286772, des8=130442, con8=135294,
               risco=[(22100, 286772), (31040, 263873), (24120, 231184), (18879, 203928)]),
}


# =========================================================================
# 4) Mesmos dados, abertos por curso: (coorte, ingressantes, formados por ano, saídas por ano)
#    Usados na reamostragem por curso (bootstrap), porque os cursos diferem muito entre si.
# =========================================================================
POR_CURSO = [
    (2011, 86, [0, 0, 0, 5, 11, 8, 10, 4], [0, 18, 13, 6, 5, 3, 1, 0]),
    (2011, 49, [0, 0, 0, 1, 1, 2, 2, 0], [3, 6, 2, 14, 1, 13, 3, 0]),
    (2011, 59, [0, 0, 0, 3, 7, 6, 8, 3], [9, 7, 8, 3, 2, 1, 0, 2]),
    (2011, 95, [0, 0, 0, 1, 7, 7, 8, 5], [9, 10, 11, 9, 22, 1, 0, 4]),
    (2011, 81, [0, 1, 1, 3, 13, 7, 3, 0], [16, 4, 13, 12, 2, 3, 3, 0]),
    (2011, 46, [0, 0, 1, 2, 4, 5, 3, 3], [1, 4, 3, 0, 2, 6, 1, 5]),
    (2011, 26, [0, 0, 0, 8, 3, 1, 3, 1], [0, 1, 2, 2, 2, 0, 2, 0]),
    (2011, 27, [0, 0, 1, 5, 2, 0, 0, 0], [2, 10, 2, 4, 0, 1, 0, 0]),
    (2011, 96, [6, 0, 1, 1, 8, 7, 9, 8], [12, 11, 6, 10, 6, 2, 1, 0]),
    (2011, 71, [0, 0, 0, 0, 5, 0, 1, 1], [0, 15, 12, 8, 8, 9, 5, 1]),
    (2011, 42, [0, 0, 0, 4, 2, 6, 6, 1], [0, 2, 0, 5, 1, 1, 2, 10]),
    (2011, 54, [0, 0, 1, 0, 0, 0, 3, 2], [8, 25, 2, 5, 3, 5, 0, 0]),
    (2011, 50, [0, 0, 0, 0, 3, 6, 3, 5], [3, 6, 7, 10, 1, 3, 1, 0]),
    (2011, 29, [0, 0, 0, 0, 5, 0, 2, 0], [1, 11, 3, 3, 1, 1, 1, 1]),
    (2011, 53, [2, 0, 0, 1, 6, 4, 6, 0], [3, 10, 13, 6, 1, 1, 0, 0]),
    (2011, 14, [1, 0, 0, 0, 0, 0, 0, 0], [3, 2, 0, 5, 0, 0, 0, 0]),
    (2011, 27, [0, 0, 0, 0, 4, 6, 1, 1], [1, 3, 2, 4, 1, 0, 0, 1]),
    (2011, 72, [0, 0, 2, 2, 6, 8, 1, 1], [25, 2, 10, 4, 5, 3, 0, 1]),
    (2011, 67, [0, 0, 0, 7, 0, 0, 0, 0], [26, 17, 8, 3, 1, 3, 0, 1]),
    (2011, 35, [0, 2, 0, 1, 1, 0, 0, 3], [0, 9, 4, 3, 5, 2, 2, 0]),
    (2011, 27, [0, 0, 0, 3, 1, 2, 0, 0], [1, 4, 12, 1, 0, 0, 2, 0]),
    (2011, 73, [1, 0, 1, 2, 1, 1, 4, 1], [11, 13, 11, 5, 8, 1, 5, 2]),
    (2011, 52, [0, 0, 0, 3, 3, 1, 0, 0], [3, 14, 9, 12, 1, 3, 1, 2]),
    (2011, 88, [0, 0, 0, 1, 9, 1, 9, 2], [30, 8, 7, 10, 3, 5, 1, 1]),
    (2011, 41, [0, 0, 0, 0, 2, 2, 2, 0], [0, 12, 15, 3, 2, 2, 1, 0]),
    (2011, 55, [0, 0, 0, 1, 5, 2, 3, 0], [8, 2, 3, 25, 3, 0, 3, 0]),
    (2011, 34, [0, 0, 0, 1, 2, 1, 0, 3], [0, 9, 4, 2, 3, 2, 0, 3]),
    (2011, 60, [0, 0, 0, 1, 2, 1, 1, 0], [17, 8, 8, 6, 6, 1, 1, 3]),
    (2011, 45, [0, 0, 0, 0, 0, 1, 1, 1], [1, 14, 9, 5, 3, 1, 2, 1]),
    (2011, 60, [0, 0, 0, 0, 3, 1, 1, 0], [0, 8, 5, 0, 2, 32, 3, 4]),
    (2011, 58, [0, 0, 0, 0, 3, 4, 4, 2], [7, 8, 7, 17, 0, 4, 0, 1]),
    (2011, 30, [0, 0, 0, 0, 0, 0, 0, 0], [10, 19, 1, 0, 0, 0, 0, 0]),
    (2011, 28, [0, 0, 1, 1, 2, 4, 4, 1], [0, 4, 3, 0, 4, 0, 1, 2]),
    (2013, 80, [0, 0, 3, 4, 10, 11, 4, 4], [2, 16, 16, 5, 1, 2, 0, 1]),
    (2013, 50, [0, 0, 0, 2, 3, 0, 2, 0], [7, 3, 1, 14, 10, 5, 2, 0]),
    (2013, 56, [0, 0, 0, 4, 7, 4, 3, 5], [10, 6, 6, 4, 4, 1, 1, 0]),
    (2013, 131, [0, 0, 0, 2, 10, 2, 5, 3], [53, 7, 18, 16, 8, 2, 2, 1]),
    (2013, 86, [0, 2, 3, 4, 10, 16, 2, 3], [16, 2, 3, 15, 5, 4, 1, 0]),
    (2013, 47, [0, 0, 1, 5, 6, 4, 0, 1], [1, 1, 6, 1, 9, 4, 5, 0]),
    (2013, 29, [0, 0, 0, 1, 2, 4, 1, 2], [2, 2, 4, 1, 5, 0, 1, 2]),
    (2013, 58, [0, 1, 1, 0, 4, 5, 8, 4], [3, 7, 5, 3, 6, 2, 2, 0]),
    (2013, 124, [0, 0, 0, 0, 10, 1, 1, 1], [41, 13, 24, 11, 6, 1, 5, 1]),
    (2013, 38, [0, 0, 0, 0, 7, 5, 1, 0], [0, 1, 2, 3, 0, 13, 0, 0]),
    (2013, 46, [0, 0, 0, 3, 4, 1, 1, 0], [16, 10, 6, 2, 3, 0, 0, 0]),
    (2013, 53, [0, 0, 0, 0, 3, 7, 4, 6], [1, 5, 5, 8, 5, 3, 4, 0]),
    (2013, 28, [0, 0, 0, 0, 1, 3, 4, 1], [4, 5, 8, 0, 1, 0, 0, 0]),
    (2013, 52, [0, 1, 0, 3, 4, 7, 3, 2], [5, 7, 6, 6, 4, 2, 0, 1]),
    (2013, 6, [0, 0, 0, 0, 0, 0, 0, 0], [0, 3, 0, 0, 0, 0, 1, 0]),
    (2013, 28, [0, 0, 0, 0, 1, 0, 4, 0], [4, 8, 3, 1, 3, 2, 0, 1]),
    (2013, 63, [0, 0, 0, 3, 5, 3, 1, 2], [1, 10, 13, 4, 11, 2, 1, 0]),
    (2013, 64, [0, 0, 0, 5, 4, 3, 2, 1], [8, 10, 17, 6, 3, 2, 2, 1]),
    (2013, 20, [0, 1, 1, 0, 0, 3, 2, 1], [1, 2, 2, 2, 0, 1, 3, 0]),
    (2013, 28, [0, 0, 1, 3, 3, 1, 0, 0], [1, 6, 0, 2, 6, 0, 0, 0]),
    (2013, 79, [0, 0, 0, 1, 0, 1, 0, 1], [13, 21, 16, 10, 5, 7, 2, 0]),
    (2013, 44, [0, 0, 0, 5, 0, 1, 0, 0], [1, 19, 7, 4, 1, 1, 1, 1]),
    (2013, 98, [0, 0, 1, 5, 4, 7, 4, 2], [32, 7, 9, 9, 12, 3, 1, 1]),
    (2013, 40, [0, 0, 0, 0, 4, 1, 1, 1], [0, 15, 9, 4, 2, 1, 1, 0]),
    (2013, 6, [0, 0, 1, 0, 0, 0, 0, 0], [1, 0, 1, 1, 0, 1, 0, 1]),
    (2013, 49, [0, 0, 0, 3, 7, 1, 0, 0], [0, 18, 9, 2, 6, 1, 1, 0]),
    (2013, 50, [0, 0, 0, 1, 0, 4, 2, 1], [13, 5, 8, 2, 0, 7, 3, 0]),
    (2013, 64, [0, 0, 0, 0, 0, 2, 6, 4], [11, 12, 7, 9, 4, 3, 2, 1]),
    (2013, 51, [0, 0, 0, 0, 0, 7, 0, 0], [1, 19, 8, 9, 1, 2, 0, 0]),
    (2013, 71, [0, 0, 0, 0, 2, 2, 1, 1], [3, 4, 2, 17, 19, 5, 8, 1]),
    (2013, 51, [0, 0, 2, 1, 3, 2, 0, 1], [9, 2, 4, 21, 3, 0, 0, 0]),
    (2013, 33, [0, 2, 0, 3, 1, 0, 0, 0], [7, 11, 7, 1, 1, 0, 0, 0]),
    (2013, 39, [0, 0, 0, 1, 1, 4, 1, 1], [0, 4, 8, 7, 2, 6, 3, 0]),
    (2015, 92, [0, 1, 0, 7, 7, 6, 3, 5], [8, 17, 13, 8, 4, 1, 0, 1]),
    (2015, 51, [0, 0, 1, 2, 5, 2, 0, 0], [4, 0, 1, 19, 6, 1, 0, 0]),
    (2015, 51, [0, 0, 0, 4, 4, 9, 1, 5], [4, 8, 5, 1, 2, 3, 0, 1]),
    (2015, 125, [0, 0, 0, 1, 7, 10, 2, 1], [52, 11, 12, 12, 4, 3, 2, 1]),
    (2015, 79, [0, 0, 1, 12, 10, 5, 1, 0], [17, 3, 17, 6, 4, 0, 0, 1]),
    (2015, 51, [0, 0, 0, 6, 6, 4, 4, 1], [2, 4, 4, 9, 7, 1, 2, 0]),
    (2015, 30, [0, 0, 0, 5, 3, 3, 0, 2], [5, 0, 7, 1, 2, 0, 1, 0]),
    (2015, 56, [0, 0, 0, 0, 1, 4, 1, 4], [1, 4, 9, 10, 3, 4, 4, 3]),
    (2015, 159, [0, 0, 0, 1, 2, 7, 4, 1], [22, 76, 13, 7, 10, 8, 2, 3]),
    (2015, 37, [0, 0, 0, 11, 2, 0, 1, 0], [2, 0, 3, 10, 0, 0, 1, 0]),
    (2015, 49, [0, 0, 0, 2, 4, 0, 2, 0], [14, 9, 9, 4, 2, 1, 0, 0]),
    (2015, 52, [0, 0, 0, 1, 5, 6, 0, 0], [1, 6, 10, 6, 4, 1, 0, 9]),
    (2015, 14, [0, 0, 0, 0, 2, 4, 0, 0], [0, 2, 1, 2, 1, 1, 1, 0]),
    (2015, 50, [0, 0, 0, 3, 7, 2, 3, 1], [16, 4, 4, 3, 4, 1, 1, 1]),
    (2015, 46, [0, 0, 0, 1, 1, 1, 5, 1], [12, 6, 1, 2, 0, 0, 1, 0]),
    (2015, 29, [0, 0, 0, 2, 1, 0, 2, 1], [3, 9, 3, 3, 2, 2, 0, 0]),
    (2015, 65, [0, 0, 1, 3, 3, 6, 1, 0], [4, 11, 11, 8, 3, 1, 2, 5]),
    (2015, 67, [0, 0, 0, 7, 5, 3, 1, 0], [9, 10, 10, 2, 1, 4, 4, 2]),
    (2015, 17, [0, 0, 0, 1, 0, 2, 0, 0], [2, 1, 3, 2, 1, 1, 0, 1]),
    (2015, 32, [0, 0, 0, 0, 6, 0, 1, 0], [0, 12, 7, 0, 1, 0, 2, 0]),
    (2015, 59, [0, 0, 0, 0, 5, 0, 1, 0], [14, 18, 9, 6, 2, 1, 1, 0]),
    (2015, 35, [0, 0, 0, 1, 1, 0, 1, 0], [1, 17, 3, 5, 1, 0, 0, 0]),
    (2015, 93, [0, 0, 1, 5, 2, 3, 9, 2], [17, 19, 11, 5, 13, 1, 1, 0]),
    (2015, 42, [0, 0, 0, 1, 5, 7, 0, 1], [0, 18, 5, 2, 2, 0, 0, 0]),
    (2015, 12, [0, 0, 1, 0, 1, 0, 1, 0], [2, 1, 0, 4, 1, 1, 0, 0]),
    (2015, 46, [0, 0, 0, 2, 2, 4, 0, 2], [0, 5, 23, 2, 3, 0, 0, 0]),
    (2015, 47, [0, 0, 1, 1, 0, 0, 1, 1], [14, 12, 1, 8, 1, 3, 0, 1]),
    (2015, 49, [0, 0, 1, 0, 0, 1, 2, 0], [21, 10, 5, 1, 2, 2, 0, 0]),
    (2015, 51, [0, 0, 0, 0, 1, 4, 3, 0], [2, 20, 5, 5, 3, 1, 0, 4]),
    (2015, 78, [0, 0, 0, 0, 2, 4, 1, 3], [2, 4, 2, 33, 10, 1, 0, 1]),
    (2015, 78, [0, 0, 0, 1, 9, 4, 3, 0], [3, 1, 21, 12, 5, 0, 7, 0]),
    (2015, 25, [0, 0, 0, 0, 2, 0, 1, 0], [12, 7, 3, 0, 0, 0, 0, 0]),
    (2015, 51, [0, 0, 0, 3, 7, 1, 3, 0], [2, 8, 11, 7, 6, 0, 0, 0]),
]

# Fator de correção ×0,80: não se aplica (ver cabeçalho).

# ---- agregados de Estatística ----
ing_est = sum(EST[c]['ing'] for c in COORTES)
des_ano = np.array([sum(EST[c]['des'][a] for c in COORTES) for a in range(T)])
con_ano = np.array([sum(EST[c]['con'][a] for c in COORTES) for a in range(T)])
risco_ano = np.array([sum(EST[c]['risco'][a] for c in COORTES) for a in range(T)])
h_est = des_ano / risco_ano                                  # risco anual de sair, Estatística

# ---- agregados do grupo comparável ----
ing_comp = sum(COMP[c]['ing'] for c in COMP)
des_comp = np.array([sum(COMP[c]['risco'][a][0] for c in COMP) for a in range(4)])
risco_comp = np.array([sum(COMP[c]['risco'][a][1] for c in COMP) for a in range(4)])
h_comp = des_comp / risco_comp

print("=" * 70)
print("  DADOS DA LITERATURA")
print("=" * 70)
print(f"\n  Stinebrickner & Stinebrickner (2014) · N = {N_berea}")
print(f"  → Chance dada na entrada ao destino que viveu:  {p_destino_real*100:.1f}%")
print(f"  → Quanto esperavam acertar:                     {p_esperava_acertar*100:.1f}%")
print(f"  → Acaso puro (1 em {n_destinos}):                         {p_acaso*100:.1f}%")
print(f"  → Ciências: davam {ciencias_entrada*100:.1f}%, formaram-se {ciencias_saida*100:.1f}%")
print(f"  → Desistência: davam {desist_esperada*100:.1f}%, desistiram {desist_real*100:.1f}%")
def mil(x):
    return f"{x:,}".replace(',', '.')

print(f"\n  INEP · Estatística · coortes {', '.join(map(str, COORTES))} · {mil(ing_est)} ingressantes")
print("  ano |  no início  | saíram | formaram | risco de sair no ano")
for a in range(T):
    print(f"   {a+1}  | {mil(risco_ano[a]):>9} | {mil(des_ano[a]):>6} | {mil(con_ano[a]):>8} | {h_est[a]*100:5.1f}%")
print(f"\n  Em 8 anos: saíram {des_ano.sum()/ing_est*100:.1f}% · formaram {con_ano.sum()/ing_est*100:.1f}%")
print(f"\n  INEP · grupo comparável (público, presencial, bacharelado) · {mil(ing_comp)} ingressantes")
print(f"  Em 8 anos: saíram {sum(COMP[c]['des8'] for c in COMP)/ing_comp*100:.1f}% · "
      f"formaram {sum(COMP[c]['con8'] for c in COMP)/ing_comp*100:.1f}%")
print("=" * 70)


In [ ]:
# --- O MODELO ---
# 1) o que se sabe na entrada · 2) a evasão comparada · 3) as oito corridas

print("=" * 70)
print("  PARTE 1 · O QUE SE SABE NA ENTRADA")
print("=" * 70)
print(f"  Acerto na entrada vs acaso:     {p_destino_real/p_acaso:.2f}× o acaso")
print(f"  Confiança vs acerto real:       {p_esperava_acertar/p_destino_real:.2f}×")
print(f"  Desistência real vs imaginada:  {desist_real/desist_esperada:.2f}×")

# =========================================================================
# PARTE 2 · A EVASÃO COMPARADA
# =========================================================================
def qui2(pares):
    tab = np.array([[x for x, n in pares], [n - x for x, n in pares]])
    return stats.chi2_contingency(tab)[1]

# Estatística: o ano 1 é diferente dos anos 2 a 4? E os anos 2 a 4 são homogêneos?
p_ano1 = qui2([(des_ano[0], risco_ano[0]), (des_ano[1:4].sum(), risco_ano[1:4].sum())])
p_hom = qui2([(des_ano[a], risco_ano[a]) for a in (1, 2, 3)])
h1_est = des_ano[0] / risco_ano[0]
h_reg_est = des_ano[1:4].sum() / risco_ano[1:4].sum()
h1_comp = des_comp[0] / risco_comp[0]
h_reg_comp = des_comp[1:4].sum() / risco_comp[1:4].sum()
p_grupos = qui2([(des_ano[1:4].sum(), risco_ano[1:4].sum()), (des_comp[1:4].sum(), risco_comp[1:4].sum())])
saida8_est, form8_est = des_ano.sum() / ing_est, con_ano.sum() / ing_est
saida8_comp = sum(COMP[c]['des8'] for c in COMP) / ing_comp
form8_comp = sum(COMP[c]['con8'] for c in COMP) / ing_comp

print("\n" + "=" * 70)
print("  PARTE 2 · A EVASÃO COMPARADA")
print("=" * 70)
print(f"  Estatística      · ano 1: {h1_est*100:4.1f}% · anos 2-4: {h_reg_est*100:4.1f}% ao ano")
print(f"  Grupo comparável · ano 1: {h1_comp*100:4.1f}% · anos 2-4: {h_reg_comp*100:4.1f}% ao ano")
print(f"\n  Razão de risco no regime: {h_reg_est/h_reg_comp:.2f}×")
print(f"  As duas populações têm o mesmo risco? p = {p_grupos:.1e}  → não")
print(f"\n  Em 8 anos: saíram {saida8_est*100:.1f}% (Estatística) contra {saida8_comp*100:.1f}% (comparável)")
print(f"             formaram {form8_est*100:.1f}% contra {form8_comp*100:.1f}%")
print(f"\n  Dentro da Estatística:")
print(f"  → o ano 1 é mais gentil que os anos 2-4:  p = {p_ano1:.1e}")
print(f"  → os anos 2-4 são homogêneos entre si:    p = {p_hom:.2f}  (risco constante)")

# =========================================================================
# PARTE 3 · AS OITO CORRIDAS
# Modelo de riscos competitivos, ajustado por máxima verossimilhança às contagens.
#   formar:  tempo até vencer k etapas ~ Gama(k, λ)
#   sair:    exponencial por trechos, taxa μ₁ no ano 1 e μ depois
# Quatro parâmetros livres: k, λ, μ₁, μ. Nada é chutado.
# =========================================================================
malha = np.linspace(0, T, 4001)

def probabilidades(k, lam, mu1, mu):
    """P(formar no ano t), P(sair no ano t) e P(ainda no curso no fim do ano 8)."""
    H_saida = np.where(malha <= 1, mu1 * malha, mu1 + mu * (malha - 1))
    S_saida = np.exp(-H_saida)                       # sobrevivência à desistência
    h_saida = np.where(malha <= 1, mu1, mu)
    f_form = stats.gamma.pdf(malha, a=k, scale=1 / lam)
    F_form = stats.gamma.cdf(malha, a=k, scale=1 / lam)
    p_form, p_saida = [], []
    for t in range(1, T + 1):
        m = (malha > t - 1) & (malha <= t)
        p_form.append(np.trapezoid(f_form[m] * S_saida[m], malha[m]))
        p_saida.append(np.trapezoid(h_saida[m] * S_saida[m] * (1 - F_form[m]), malha[m]))
    resto = max(1e-12, S_saida[-1] * (1 - F_form[-1]))
    return np.array(p_form), np.array(p_saida), resto

def menos_log_verossim(par):
    k, lam, mu1, mu = np.exp(par)
    if k > 40:
        return 1e9
    p_form, p_saida, resto = probabilidades(k, lam, mu1, mu)
    total = p_form.sum() + p_saida.sum() + resto
    p_form, p_saida, resto = p_form / total, p_saida / total, resto / total
    ainda = ing_est - con_ano.sum() - des_ano.sum()
    return -(np.sum(con_ano * np.log(np.maximum(p_form, 1e-12)))
             + np.sum(des_ano * np.log(np.maximum(p_saida, 1e-12)))
             + ainda * np.log(resto))

melhor = None
for k0 in (4, 6, 8, 10, 14):                        # vários pontos de partida
    r = optimize.minimize(menos_log_verossim, np.log([k0, 1.2, 0.14, 0.22]),
                          method='Nelder-Mead', options=dict(maxiter=8000, xatol=1e-4, fatol=1e-3))
    if melhor is None or r.fun < melhor.fun:
        melhor = r
k_aj, lam_aj, mu1_aj, mu_aj = np.exp(melhor.x)

p_form, p_saida, resto = probabilidades(k_aj, lam_aj, mu1_aj, mu_aj)
total = p_form.sum() + p_saida.sum() + resto
p_form, p_saida, resto = p_form / total, p_saida / total, resto / total
p_corrida = lam_aj / (lam_aj + mu_aj)               # chance de vencer UMA etapa
p_formula = p_corrida ** k_aj                       # a fórmula fechada

print("\n" + "=" * 70)
print("  PARTE 3 · AS OITO CORRIDAS")
print("=" * 70)
print(f"  Ajuste por máxima verossimilhança (4 parâmetros, {ing_est} alunos):")
print(f"  → k  = {k_aj:.2f} etapas")
print(f"  → λ  = {lam_aj:.3f} etapas por ano  (uma etapa a cada {12/lam_aj:.1f} meses)")
print(f"  → μ₁ = {mu1_aj:.3f} por ano (ano 1) · μ = {mu_aj:.3f} por ano (depois)")
print(f"  → tempo médio até as {k_aj:.0f} etapas: {k_aj/lam_aj:.1f} anos")

print(f"\n  Conferência 1 · as taxas de saída ajustadas contra a estimativa direta:")
print(f"  → ano 1: {mu1_aj:.3f} (ajuste) vs {-np.log(1-h1_est):.3f} (contagem direta)")
print(f"  → depois: {mu_aj:.3f} (ajuste) vs {-np.log(1-h_reg_est):.3f} (contagem direta)")

print(f"\n  Conferência 2 · perfil de verossimilhança em k (inteiros):")
for kk in range(6, 11):
    f = lambda p: menos_log_verossim(np.concatenate([[np.log(kk)], p]))
    rr = optimize.minimize(f, np.log([1.2, 0.15, 0.22]), method='Nelder-Mead', options=dict(maxiter=6000))
    print(f"   k = {kk:>2}: -logL = {rr.fun:8.1f}{'   ← melhor' if abs(kk - round(k_aj)) < 0.5 else ''}")

print(f"\n  A CORRIDA:")
print(f"  → chance de vencer UMA etapa: λ/(λ+μ) = {p_corrida*100:.1f}%")
print(f"  → {p_corrida:.3f} elevado a {k_aj:.0f} = {p_formula*100:.1f}%")
print(f"  → modelo completo (com o ano 1 mais gentil): {p_form.sum()*100:.1f}% formam")
print(f"  → observado no INEP:                         {form8_est*100:.1f}% formaram")

print(f"\n  Previsto × observado, ano a ano (% dos ingressantes):")
print("  ano | formados obs/mod | saídas obs/mod")
for t in range(T):
    print(f"   {t+1}  |   {con_ano[t]/ing_est*100:5.1f} / {p_form[t]*100:5.1f}   |  {des_ano[t]/ing_est*100:5.1f} / {p_saida[t]*100:5.1f}")

esperado = np.concatenate([p_form, p_saida, [resto]]) * ing_est
observado = np.concatenate([con_ano, des_ano, [ing_est - con_ano.sum() - des_ano.sum()]])
chi = np.sum((observado - esperado) ** 2 / np.maximum(esperado, 1e-9))
gl = len(observado) - 1 - 4
print(f"\n  Aderência: χ² = {chi:.0f}, gl = {gl}, p = {1-stats.chi2.cdf(chi, gl):.4f}")
print("  O ajuste acerta o agregado e o desenho ano a ano, mas é rejeitado como ajuste exato.")
print("  Isso é esperado: um único par (k, λ) não descreve 33 cursos diferentes entre si.")

# =========================================================================
# PARTE 4 · CHECAGENS
# Lista definida ANTES de rodar, e relatada inteira: as que passam, a que rejeita
# e a que fica no limiar.
#   (a) incerteza dos parâmetros com reamostragem POR CURSO
#   (b) previsão fora da amostra: ajusta em 2011 e 2013, prevê 2015
#   (c) sensibilidade ao corte do acompanhamento (ano 7 em vez de ano 8)
#   (d) o ritmo das etapas é mesmo constante? (Gama contra Gama generalizada)
# =========================================================================
def ajusta(con, des, ing, Tmax=T, inicio=None):
    """Repete o ajuste da Parte 3 para um subconjunto qualquer de dados."""
    def alvo(par):
        k, lam, mu1, mu = np.exp(par)
        if k > 40:
            return 1e9
        pf, ps, r = probabilidades(k, lam, mu1, mu)
        pf, ps = pf[:Tmax], ps[:Tmax]
        if Tmax < T:                                  # recalcula quem ainda está no curso
            r = 1 - pf.sum() - ps.sum()
        total = pf.sum() + ps.sum() + r
        pf, ps, r = pf / total, ps / total, max(r / total, 1e-12)
        ainda = ing - con[:Tmax].sum() - des[:Tmax].sum()
        return -(np.sum(con[:Tmax] * np.log(np.maximum(pf, 1e-12)))
                 + np.sum(des[:Tmax] * np.log(np.maximum(ps, 1e-12))) + ainda * np.log(r))
    r = optimize.minimize(alvo, np.log(inicio if inicio is not None else [8, 1.2, 0.15, 0.22]),
                          method='Nelder-Mead', options=dict(maxiter=6000, fatol=1e-3, xatol=1e-3))
    return np.exp(r.x)

print("\n" + "=" * 70)
print("  PARTE 4 · CHECAGENS")
print("=" * 70)

# (a) reamostragem por curso
N_BOOT = 150
indices = np.arange(len(POR_CURSO))
amostras = []
for _ in range(N_BOOT):
    escolha = rng.choice(indices, size=len(indices), replace=True)
    ing_b = sum(POR_CURSO[i][1] for i in escolha)
    con_b = np.sum([POR_CURSO[i][2] for i in escolha], axis=0).astype(float)
    des_b = np.sum([POR_CURSO[i][3] for i in escolha], axis=0).astype(float)
    amostras.append(ajusta(con_b, des_b, ing_b, inicio=[k_aj, lam_aj, mu1_aj, mu_aj]))
amostras = np.array(amostras)
print(f"  (a) Reamostragem por curso ({N_BOOT} reamostras de {len(POR_CURSO)} cursos):")
for j, nome in enumerate(['k ', 'λ ', 'μ₁', 'μ ']):
    lo, hi = np.percentile(amostras[:, j], [2.5, 97.5])
    print(f"      {nome}: IC95% [{lo:.3f}, {hi:.3f}]")
print(f"      k arredondado cai em 8 em {np.mean(np.round(amostras[:, 0]) == 8)*100:.0f}% das reamostras.")
print(f"      Leitura honesta: k está entre 7 e 9, e não 'exatamente 8'.")

# (b) fora da amostra
con_tr = np.array([EST[c]['con'] for c in (2011, 2013)], dtype=float).sum(axis=0)
des_tr = np.array([EST[c]['des'] for c in (2011, 2013)], dtype=float).sum(axis=0)
ing_tr = EST[2011]['ing'] + EST[2013]['ing']
par_tr = ajusta(con_tr, des_tr, ing_tr)
pf_tr, ps_tr, r_tr = probabilidades(*par_tr)
tot_tr = pf_tr.sum() + ps_tr.sum() + r_tr
ing_te = EST[2015]['ing']
form_te = np.sum(EST[2015]['con']) / ing_te
saida_te = np.sum(EST[2015]['des']) / ing_te
print(f"\n  (b) Fora da amostra (ajusta em 2011 e 2013, prevê 2015, que o modelo não viu):")
print(f"      k = {par_tr[0]:.2f} · λ = {par_tr[1]:.2f}")
print(f"      formados: previsto {pf_tr.sum()/tot_tr*100:.1f}% · observado {form_te*100:.1f}%")
print(f"      saídas:   previsto {ps_tr.sum()/tot_tr*100:.1f}% · observado {saida_te*100:.1f}%")

# (c) sensibilidade ao corte
par7 = ajusta(con_ano.astype(float), des_ano.astype(float), ing_est, Tmax=7)
print(f"\n  (c) Corte no ano 7 em vez do ano 8: k = {par7[0]:.2f} · λ = {par7[1]:.3f}"
      f" (contra {k_aj:.2f} e {lam_aj:.3f})")

# (d) o ritmo das etapas é constante?
def menos_log_verossim_gg(par):
    k, lam, c, mu1, mu = np.exp(par)
    if k > 40 or c > 6:
        return 1e9
    H = np.where(malha <= 1, mu1 * malha, mu1 + mu * (malha - 1))
    S = np.exp(-H); h = np.where(malha <= 1, mu1, mu)
    f = stats.gengamma.pdf(malha, a=k, c=c, scale=1 / lam)
    F = stats.gengamma.cdf(malha, a=k, c=c, scale=1 / lam)
    pf, ps = [], []
    for t in range(1, T + 1):
        m = (malha > t - 1) & (malha <= t)
        pf.append(np.trapezoid(f[m] * S[m], malha[m]))
        ps.append(np.trapezoid(h[m] * S[m] * (1 - F[m]), malha[m]))
    pf, ps = np.array(pf), np.array(ps)
    resto_gg = max(1e-12, S[-1] * (1 - F[-1]))
    total = pf.sum() + ps.sum() + resto_gg
    pf, ps, resto_gg = pf / total, ps / total, resto_gg / total
    ainda = ing_est - con_ano.sum() - des_ano.sum()
    return -(np.sum(con_ano * np.log(np.maximum(pf, 1e-12)))
             + np.sum(des_ano * np.log(np.maximum(ps, 1e-12))) + ainda * np.log(resto_gg))

melhor_gg = None
for inicio in ([8, 1.2, 1, 0.15, 0.22], [4, 1, 1.5, 0.15, 0.22]):
    rgg = optimize.minimize(menos_log_verossim_gg, np.log(inicio),
                            method='Nelder-Mead', options=dict(maxiter=20000, fatol=1e-4))
    if melhor_gg is None or rgg.fun < melhor_gg.fun:
        melhor_gg = rgg
lr = 2 * (melhor.fun - melhor_gg.fun)
print(f"\n  (d) Ritmo das etapas constante (Gama) contra ritmo variável (Gama generalizada):")
print(f"      razão de verossimilhança χ² = {lr:.2f}, gl = 1, p = {1-stats.chi2.cdf(lr,1):.3f}")
print("      Os dados agregados NÃO distinguem ritmo constante de ritmo levemente crescente.")
print("      A ideia de que as disciplinas da área se concentram no fim do curso fica plausível")
print("      e não testada por estes dados.")
print("\n  Não testável com dados agregados: a independência entre os dois relógios.")
print("  Quem está travando numa disciplina também pode estar mais perto de desistir, e")
print("  riscos competitivos não separam isso sem dados individuais. Limitação estrutural.")
print("=" * 70)


In [ ]:
# --- VISUALIZAÇÃO ---
rodape = lambda txt: plt.figtext(0.5, 0.005, txt + ' | #365Probabilidades', ha='center', fontsize=9, color='gray')

# ── GRÁFICO 1 · O que se sabe na entrada ──
fig1, (a, b) = plt.subplots(1, 2, figsize=(12, 6), gridspec_kw={'width_ratios': [1.1, 1]})
vals = [p_acaso, p_destino_real, p_esperava_acertar]
labs = ['Acaso puro\n(1 em 8)', 'Chance dada ao\ndestino que viveu', 'Quanto esperavam\nacertar']
barras = a.bar(labs, np.array(vals) * 100, color=[CINZA, VERDE, DOURADO], width=0.55)
for barra, v in zip(barras, vals):
    a.text(barra.get_x() + barra.get_width() / 2, v * 100 + 1, f'{v*100:.1f}%',
           ha='center', fontweight='bold', fontsize=14)
a.set_ylim(0, 55); a.set_ylabel('%')
a.set_title('Na entrada, cada aluno apostou no próprio destino', fontsize=12)
x = np.arange(2); w = 0.36
b.bar(x - w/2, [ciencias_entrada*100, desist_esperada*100], w, color=DOURADO, label='Chance média dada na entrada')
b.bar(x + w/2, [ciencias_saida*100, desist_real*100], w, color=VERMELHO, label='O que aconteceu')
for xi, (e, s) in enumerate([(ciencias_entrada, ciencias_saida), (desist_esperada, desist_real)]):
    b.text(xi - w/2, e*100 + 1, f'{e*100:.1f}%', ha='center', fontsize=11)
    b.text(xi + w/2, s*100 + 1, f'{s*100:.1f}%', ha='center', fontsize=11)
b.set_xticks(x); b.set_xticklabels(['Formar-se em\nciências', 'Desistir'])
b.set_ylim(0, 45); b.legend(frameon=False)
b.set_title('O que imaginavam e o que aconteceu', fontsize=12)
fig1.suptitle(f'Ninguém escolhe sabendo · Stinebrickner & Stinebrickner (2014), N = {N_berea}', fontsize=13)
rodape('Fonte: Stinebrickner & Stinebrickner, Review of Economic Studies, 2014 (NBER WP 19165)')
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.savefig('dia-100-grafico-01-o-que-sabiam.png', dpi=150, bbox_inches='tight'); plt.close()
print("✅ Gráfico 1 salvo!")

# ── GRÁFICO 2 · Estatística contra o grupo comparável ──
fig2, (a, b) = plt.subplots(1, 2, figsize=(13, 6), gridspec_kw={'width_ratios': [1.2, 1]})
anos4 = np.arange(1, 5); w = 0.38
a.bar(anos4 - w/2, h_est[:4]*100, w, color=VERMELHO, label='Estatística')
a.bar(anos4 + w/2, h_comp*100, w, color=CINZA, label='Público, presencial, bacharelado')
for t in range(4):
    a.text(anos4[t] - w/2, h_est[t]*100 + 0.4, f'{h_est[t]*100:.1f}', ha='center', fontsize=10)
    a.text(anos4[t] + w/2, h_comp[t]*100 + 0.4, f'{h_comp[t]*100:.1f}', ha='center', fontsize=10)
a.set_xticks(anos4); a.set_xlabel('Ano desde a entrada')
a.set_ylabel('Risco de sair do curso no ano (%)'); a.set_ylim(0, 26)
a.legend(frameon=False, loc='upper left')
a.set_title(f'O relógio da saída corre {h_reg_est/h_reg_comp:.1f}× mais rápido em Estatística', fontsize=12)
grupos = ['Estatística', 'Público, presencial,\nbacharelado']
saidas = [saida8_est*100, saida8_comp*100]
formados = [form8_est*100, form8_comp*100]
ainda = [100 - saidas[i] - formados[i] for i in range(2)]
b.bar(grupos, saidas, color='#dcaaa4', label='saíram do curso', width=0.5)
b.bar(grupos, formados, bottom=saidas, color=VERDE, label='se formaram', width=0.5)
b.bar(grupos, ainda, bottom=np.array(saidas)+np.array(formados), color='#e8e6e0',
      label='ainda no curso', width=0.5)
for i in range(2):
    b.text(i, saidas[i]/2, f'{saidas[i]:.0f}%', ha='center', va='center', fontsize=14, fontweight='bold')
    b.text(i, saidas[i] + formados[i]/2, f'{formados[i]:.0f}%', ha='center', va='center', fontsize=12)
b.set_ylim(0, 100); b.set_ylabel('% dos ingressantes')
b.legend(frameon=False, loc='upper center', bbox_to_anchor=(0.5, -0.11), ncol=3, fontsize=9)
b.set_title('Onde a turma está no ano 8', fontsize=12)
fig2.suptitle('Estatística contra o grupo exatamente comparável · INEP, coortes 2011, 2013 e 2015', fontsize=13)
rodape('Fonte: INEP, Indicadores de Trajetória · comparável: todo bacharelado público presencial do país')
plt.tight_layout(rect=[0, 0.03, 1, 0.94])
plt.savefig('dia-100-grafico-02-evasao-comparada.png', dpi=150, bbox_inches='tight'); plt.close()
print("✅ Gráfico 2 salvo!")

# ── GRÁFICO 3 · As oito corridas ──
fig3, (a, b) = plt.subplots(1, 2, figsize=(13.5, 6))
t_ = np.linspace(0, 12, 700)
a.plot(t_, stats.gamma.pdf(t_, a=k_aj, scale=1/lam_aj), color=VERDE, lw=2.6,
       label=f'formar: Gama(k={k_aj:.1f}, λ={lam_aj:.2f})')
a.fill_between(t_, stats.gamma.pdf(t_, a=k_aj, scale=1/lam_aj), alpha=0.18, color=VERDE)
a.plot(t_, stats.expon.pdf(t_, scale=1/mu_aj), color=VERMELHO, lw=2.6,
       label=f'sair: Exponencial(μ={mu_aj:.2f})')
a.fill_between(t_, stats.expon.pdf(t_, scale=1/mu_aj), alpha=0.15, color=VERMELHO)
a.axvline(k_aj/lam_aj, color=CINZA, ls='--')
a.text(k_aj/lam_aj + 0.2, a.get_ylim()[1]*0.75, f'{k_aj/lam_aj:.1f} anos\n(tempo médio\naté as {k_aj:.0f} etapas)', fontsize=10)
a.set_xlabel('anos desde a entrada'); a.set_ylabel('densidade')
a.legend(frameon=False, loc='upper right', fontsize=10)
a.set_title('Os dois relógios, ajustados aos dados', fontsize=12)
a.text(7.1, a.get_ylim()[1]*0.52,
       r'$P=\left(\dfrac{\lambda}{\lambda+\mu}\right)^{k}$' +
       f'\n{p_corrida*100:.1f}% por etapa\n{p_corrida:.3f}' + r'$^{8}$' + f' = {p_formula*100:.1f}%', fontsize=13)
anos8 = np.arange(1, T+1); w = 0.4
b.bar(anos8 - w/2, con_ano/ing_est*100, w, color=VERDE, label='formados, observado')
b.bar(anos8 + w/2, des_ano/ing_est*100, w, color='#dcaaa4', label='saídas, observado')
b.plot(anos8 - w/2, p_form*100, 'o', color='#14403c', ms=7, label='formados, modelo')
b.plot(anos8 + w/2, p_saida*100, 'o', color=VERMELHO, ms=7, label='saídas, modelo')
b.set_xticks(anos8); b.set_xlabel('Ano desde a entrada'); b.set_ylabel('% dos ingressantes')
b.legend(frameon=False, fontsize=9.5)
b.set_title(f'O modelo prevê {p_form.sum()*100:.1f}% de formados · o INEP registrou {form8_est*100:.1f}%', fontsize=12)
fig3.suptitle('Oito corridas contra a desistência: cada etapa do curso é uma corrida nova', fontsize=13)
rodape('Ajuste por máxima verossimilhança nas contagens do INEP · 5.310 alunos, coortes 2011, 2013 e 2015')
plt.tight_layout(rect=[0, 0.03, 1, 0.94])
plt.savefig('dia-100-grafico-03-oito-corridas.png', dpi=150, bbox_inches='tight'); plt.close()
print("✅ Gráfico 3 salvo!")


### 💡 O Insight

**Na entrada, o aluno médio dava 31% de chance ao destino que de fato viveu.**

Mais que o acaso, que daria 12,5%. Menos do que ele mesmo esperava acertar, 43,3%. A escolha é feita por alguém que ainda não conhece o que está escolhendo, e que nem sabe o quanto não conhece.

E o curso escolhido às cegas pode ser dos que mais perdem gente.

Em Estatística, no Brasil, **70,7% dos ingressantes saíram do curso em oito anos, contra 45,4% do grupo exatamente comparável**: todo o bacharelado público presencial do país, com mais de 560 mil alunos nas mesmas turmas. O risco anual de sair, depois do primeiro ano, é de 19,7% contra 10,7%. **O relógio da saída corre quase duas vezes mais rápido.**

Dentro da estatística, esse relógio tem um desenho. O primeiro ano é o mais gentil, com 14,0%. Depois o risco fica constante, em torno de 20% ao ano, e um risco que não muda com o tempo decorrido é a marca de um relógio sem memória. Aguentar três anos não torna o quarto mais fácil.

Contra esse relógio corre outro: o de se formar.

Formar não é um evento, é uma soma de etapas. Pedi ao modelo que estimasse **quantas etapas o curso exige** e **a que ritmo elas são vencidas**, usando apenas as contagens de formados e desistentes ano a ano. Nenhuma informação sobre como funciona um curso entrou na conta.

**O modelo encontrou 8 etapas.**

Oito, o número de semestres de um bacharelado. A cada 10 meses, em média, uma delas é vencida, e é por isso que o tempo médio até a formatura sai em 6,8 anos, não em 4.

O número exige uma ressalva que é parte do achado: reamostrando os cursos, **k fica entre 7 e 9**. O ponto central é 8, e a afirmação honesta é a faixa, não o número redondo.

E aí a conta fica simples e bonita:

**Cada etapa é uma corrida contra a vontade de sair, e você ganha cada uma com 84,4% de chance. Ganhar oito seguidas dá 26%.**

O modelo completo prevê 23,9% de formados. O INEP registrou 23,7%. As taxas de saída que o ajuste encontrou sozinho, 0,151 no primeiro ano e 0,217 depois, são as mesmas que a contagem direta dá: 0,151 e 0,219.

E o teste mais duro, o de prever o que não viu: ajustado apenas nas turmas de 2011 e 2013, o modelo prevê 24,5% de formados para a turma de 2015, que registrou 22,0%. Erra por dois pontos e meio, para mais. É bom, e não é perfeito.

Nada disso diz que quem saiu errou. Na régua do INEP, trocar de curso conta como desistência, e parte dessas pessoas foi descobrir quem era em outro lugar.

O que os dados dizem é outra coisa: **ninguém sabe quem é na hora de escolher.** O primeiro ano é o ciclo básico, e a matéria que fez a pessoa escolher o curso costuma vir depois, quando o risco de desistir já subiu. Quem vira alguém, vira ganhando uma corrida por vez.

*Quantas corridas você venceu sem perceber que estava correndo?*

---

### ⚠️ Limitações do Modelo

- **"Etapa" é uma construção do modelo.** O que k faz é controlar a forma da distribuição dos tempos de formatura. O valor encontrado (8, com faixa de 7 a 9 na reamostragem por curso) é compatível com os oito semestres de um bacharelado, mas o dado não mede disciplinas cursadas. A coincidência é sugestiva, não prova.
- **A Gama supõe que as etapas são independentes e sem memória entre si**, o que é falso no mundo real: existe pré-requisito, o semestre tem duração quase fixa e as demoras de um mesmo aluno se correlacionam. Ela foi escolhida por ajustar melhor que Weibull, lognormal e Gama deslocada, não por ser descrição literal de um curso.
- **A independência entre os dois relógios não é testável aqui.** Quem está travando numa disciplina pode estar também mais perto de desistir, e riscos competitivos não separam isso sem dados individuais. É limitação estrutural, não checagem esquecida.
- **O ritmo das etapas pode não ser constante.** O teste contra uma família com ritmo variável fica no limiar (p = 0,08): os dados agregados não distinguem as duas hipóteses. A ideia de que as disciplinas da área se concentram no fim do curso é plausível e permanece não testada.
- **O teste formal de aderência rejeita o ajuste exato** (χ² = 697, gl = 12, p < 0,001). Era esperado: um único par (k, λ) não descreve 33 cursos que diferem muito entre si. O modelo acerta o agregado e o desenho ano a ano, não cada curso.
- **O risco constante vale depois do primeiro ano** (anos 2 a 4 homogêneos, p = 0,23). O ano 1 é diferente e entra no modelo com taxa própria.
- **INEP:** trocar de curso conta como desistência, mesmo na mesma instituição. Os dados cobrem apenas cursos presenciais. O acompanhamento aqui vai até o ano 8, e quem ainda estava matriculado entra como tal, sem imputação.
- **O grupo de comparação** é todo o bacharelado público presencial, o que inclui cursos de prazos e perfis muito diferentes. Ele responde "a estatística perde mais que o bacharelado público típico", não "a estatística é o pior curso".
- **Stinebrickner & Stinebrickner:** uma faculdade americana (Berea College), voltada a estudantes de baixa renda e com bolsa integral, turmas de 2000 e 2001. A categoria "ciências" inclui matemática e não isola a estatística. Números da versão NBER Working Paper.
- **As duas fontes descrevem populações diferentes e não se somam.** Berea sustenta a incerteza na entrada; o INEP, os dois relógios.

*A ciência é honesta sobre o que não sabe. O modelo também.*

---

### 📎 Links
- Substack: [link do post]
- Instagram: [link do post]

---
*365 Probabilidades · Decidindo com dados, um dia de cada vez.*
